# Stitch — cutout-head coherence probe

Closes the one open item in the Stitch vet. **Position control already passes**; the sample shows
box-fill *slabs* + duplicated geometry, root-caused to the cutout head `(14,20)` not isolating the
object silhouette in our diffusers FLUX build. `cutout_eta` was already shown NOT to be the lever.

This notebook sweeps the **untested** levers via the block's public API — `cutout_head`, then
`region_bind_steps` (S), then a focused `cutout_eta` — and scores each on **in-box CLIP** (object is
right) and a **box-edge seam ratio** (composite doesn't hard-seam). Decision at the end:
*clean-ship a config* / *Region-Binding-only fallback* / *documented v1*.

Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
import torch
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))

## 1 · Load (load-only — the private repo is already populated by the maintainer)

In [ ]:
from diffusers import ModularPipeline
pipe = ModularPipeline.from_pretrained("remyxai/stitch-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "StitchBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)
pipe.load_components(dtype=DT); pipe.to(DEV)

## 2 · Fixed layouts + metrics (robust CLIP helper, box-edge seam ratio)

In [ ]:
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from transformers import CLIPModel, CLIPProcessor

HW = dict(height=1024, width=1024, num_inference_steps=28, guidance_scale=3.5)

# normalized [x0,y0,x1,y1] boxes; a clear left/right (and a centre for 3-obj)
LAYOUTS = {
  "2obj": {"prompt": "a red cube to the left of a blue sphere, studio product photo",
           "regions": [{"box":[0.04,0.28,0.46,0.90], "prompt":"a red cube"},
                       {"box":[0.54,0.28,0.96,0.90], "prompt":"a blue sphere"}]},
  "3obj": {"prompt": "a red cube, a green cone, and a blue sphere on a table",
           "regions": [{"box":[0.02,0.30,0.34,0.90], "prompt":"a red cube"},
                       {"box":[0.36,0.24,0.64,0.90], "prompt":"a green cone"},
                       {"box":[0.66,0.30,0.98,0.90], "prompt":"a blue sphere"}]},
}

_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
def clip_score(img, text):
    with torch.no_grad():
        ii = _proc(images=img, return_tensors="pt").to(DEV)
        ti = _proc(text=[text], return_tensors="pt", padding=True, truncation=True).to(DEV)
        out = _clip(pixel_values=ii["pixel_values"], input_ids=ti["input_ids"],
                    attention_mask=ti["attention_mask"])          # full forward -> tensors
        il, tl = out.image_embeds, out.text_embeds
        il = il / il.norm(dim=-1, keepdim=True); tl = tl / tl.norm(dim=-1, keepdim=True)
        return float((il @ tl.T).squeeze())

def crop_box(img, box):
    w, h = img.size
    return img.crop((int(box[0]*w), int(box[1]*h), int(box[2]*w), int(box[3]*h)))

def seam_ratio(img, regions):
    """max gradient energy on the vertical box borders / image-median gradient (>~3 = visible seam)."""
    g = np.asarray(img.convert("L"), np.float32)
    gx = np.abs(np.diff(g, axis=1)); med = float(np.median(gx)) + 1e-6
    W = img.size[0]; peak = 0.0
    xs = sorted({int(r["box"][0]*W) for r in regions} | {int(r["box"][2]*W) for r in regions})
    for x in xs:
        x = max(2, min(W-3, x)); peak = max(peak, float(gx[:, x-2:x+2].mean()))
    return peak / med

def region_clip(img, regions):
    return float(np.mean([clip_score(crop_box(img, r["box"]), r["prompt"]) for r in regions]))

def label_grid(pairs, cell=384):
    n=len(pairs); row=Image.new("RGB",(n*cell+(n+1)*8, cell+34),"white"); d=ImageDraw.Draw(row)
    try: F=ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",20)
    except Exception: F=ImageFont.load_default()
    for i,(name,im) in enumerate(pairs):
        x=8+i*(cell+8); row.paste(im.resize((cell,cell)),(x,0)); d.text((x+6,cell+6),name,fill="black",font=F)
    return row

def run(layout, cutout_head=(14,20), cutout_eta=0.95, region_bind_steps=10, guidance_scale=3.5, seed=0):
    lay = LAYOUTS[layout]; g = torch.Generator(DEV).manual_seed(seed)
    return pipe(prompt=lay["prompt"], regions=lay["regions"], cutout_head=cutout_head,
                cutout_eta=cutout_eta, region_bind_steps=region_bind_steps,
                height=HW["height"], width=HW["width"], num_inference_steps=HW["num_inference_steps"],
                guidance_scale=guidance_scale, generator=g).images[0]
print("helpers ready")

## 3 · Baseline (the known box-slab: default head (14,20), eta 0.95, S 10)

In [ ]:
base = run("2obj")
base.save("probe_baseline.png")
print(f"baseline 2obj  in-box CLIP={region_clip(base, LAYOUTS['2obj']['regions']):.4f}  "
      f"seam={seam_ratio(base, LAYOUTS['2obj']['regions']):.1f}")
from IPython.display import display; display(base.resize((512,512)))

## 4 · Head sweep — the untested lever
Different cutout heads at a tighter `eta=0.7`. If a head isolates the object silhouette (not the box),
its objects come out clean and its **seam ratio drops** while **in-box CLIP holds/rises**. FLUX.1-dev
has 24 heads across the single blocks; this samples a spread of heads at block 14 plus two neighbour
blocks. If **every** head still slabs, that is itself the answer (→ Region-Binding-only / v1).

In [ ]:
HEADS = [(14,2),(14,8),(14,14),(14,20),(14,23),(10,20),(18,20)]
rows, results = [], []
for bh in HEADS:
    im = run("2obj", cutout_head=bh, cutout_eta=0.7)
    im.save(f"probe_head_{bh[0]}_{bh[1]}.png")
    c, s = region_clip(im, LAYOUTS["2obj"]["regions"]), seam_ratio(im, LAYOUTS["2obj"]["regions"])
    results.append((bh, c, s)); rows.append((f"({bh[0]},{bh[1]}) C{c:.2f} S{s:.0f}", im))
    print(f"  head {bh}: in-box CLIP={c:.4f}  seam={s:.1f}")
display(label_grid(rows))
best = min(results, key=lambda r: (r[2], -r[1]))   # low seam, then high CLIP
BEST_HEAD = best[0]
print(f"\n[HEAD] cleanest = {BEST_HEAD}  (seam={best[2]:.1f}, CLIP={best[1]:.4f})")

## 5 · Region-bind-steps (S) sweep at the best head
Fewer confined steps → the unconstrained refine has more freedom to de-boxify each object. Trade-off:
too few and position weakens.

In [ ]:
rows, results = [], []
for S in [4, 6, 8, 10]:
    im = run("2obj", cutout_head=BEST_HEAD, cutout_eta=0.7, region_bind_steps=S)
    c, s = region_clip(im, LAYOUTS["2obj"]["regions"]), seam_ratio(im, LAYOUTS["2obj"]["regions"])
    results.append((S, c, s)); rows.append((f"S{S} C{c:.2f} S{s:.0f}", im))
    print(f"  S={S}: in-box CLIP={c:.4f}  seam={s:.1f}")
display(label_grid(rows))
BEST_S = min(results, key=lambda r: (r[2], -r[1]))[0]
print(f"\n[S] best = {BEST_S}")

## 6 · eta refine at (best head, best S)

In [ ]:
rows = []
for eta in [0.5, 0.65, 0.8]:
    im = run("2obj", cutout_head=BEST_HEAD, cutout_eta=eta, region_bind_steps=BEST_S)
    c, s = region_clip(im, LAYOUTS["2obj"]["regions"]), seam_ratio(im, LAYOUTS["2obj"]["regions"])
    rows.append((f"eta{eta} C{c:.2f} S{s:.0f}", im)); print(f"  eta={eta}: CLIP={c:.4f}  seam={s:.1f}")
display(label_grid(rows))

## 7 · Confirm on 3-object at the winning config + verdict

In [ ]:
BEST = dict(cutout_head=BEST_HEAD, cutout_eta=0.65, region_bind_steps=BEST_S)
print("winning config:", BEST)
for layout in ["2obj", "3obj"]:
    im = run(layout, **BEST); im.save(f"probe_final_{layout}.png")
    c, s = region_clip(im, LAYOUTS[layout]["regions"]), seam_ratio(im, LAYOUTS[layout]["regions"])
    print(f"  {layout}: in-box CLIP={c:.4f}  seam={s:.1f}")
    display(im.resize((512,512)))

base_seam = seam_ratio(base, LAYOUTS["2obj"]["regions"])
final = Image.open("probe_final_2obj.png"); fin_seam = seam_ratio(final, LAYOUTS["2obj"]["regions"])
print("\n================ VERDICT ================")
if fin_seam < 3.0 and fin_seam < base_seam:
    print(f"CLEAN-SHIP: seam {base_seam:.1f} -> {fin_seam:.1f} at {BEST}. Update the block defaults + hero.")
elif fin_seam < base_seam:
    print(f"IMPROVED but not seamless ({base_seam:.1f}->{fin_seam:.1f}). Consider documented-v1 with these defaults.")
else:
    print("NO head/S/eta cleaned the slabs -> Region-Binding-only fallback (drop Cutout/composite), or ship v1 "
          "with an explicit 'rectangular-bias / best at 2-3 objects' caveat.")